In [ ]:
pip install langchain sentence-transformers tqdm qdrant-client pandas numpy

In [ ]:
pip install langchain_community

In [ ]:
pip install -U langchain-openai

In [ ]:
!pip install --upgrade openai

In [100]:
# Importando bibliotecas necessárias
import os
from langchain_openai import OpenAI
from langchain.chains import StuffDocumentsChain
from langchain.prompts import PromptTemplate
from langchain.chains.llm import LLMChain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import qdrant_client
from qdrant_client.models import PointStruct, VectorParams, Distance
import pandas as pd
import numpy as np
import json
import logging
import concurrent.futures
import re
from collections import Counter

# **Configurar logging para debugging**

# **Carregar dados JSON**

In [ ]:
# Upload de arquivos no pc local
from google.colab import files
uploaded = files.upload()

In [ ]:
import json

# Carrega o arquivo JSON (substitua pelo caminho correto do arquivo JSON)
with open('/content/data_cpp.json') as f:
    posts_data = json.load(f)

# Visualiza o conteúdo do arquivo JSON, incluindo Body, CreationDate, Score e ViewCount
for post in posts_data[:5]:  # Exibe os primeiros 5 posts
    print(f"Post ID: {post['postId']}")
    print(f"Title: {post.get('title', 'No Title')}")
    print(f"Tag: {post.get('tagName', 'No Tag')}")
    print(f"Body: {post.get('body', 'No Body')}")
    print(f"Creation Date: {post.get('creationDate', 'No Creation Date')}")
    print(f"Score: {post.get('score', 'No Score')}")
    print(f"View Count: {post.get('viewCount', 'No View Count')}")
    print('-' * 50)

# **Criar DataFrame**

In [9]:
# Cria um DataFrame a partir dos dados JSON
df = pd.DataFrame(posts_data)

# Substituir valores NaN por strings vazias em todas as colunas
df = df.fillna('')

# Garantir que todos os valores no DataFrame sejam strings
df = df.astype(str)

In [10]:
# Preparar os textos usando o título e o corpo da pergunta
texts = []

for index, row in df.iterrows():
    if row['body']:  # Verifica se o corpo não está vazio
        texts.append(f"{row['title']} {row['body']}")
    else:
        print(f"Post ID {row['postId']} não possui corpo.")

# Visualiza os primeiros 5 textos gerados
for text in texts[:5]:
    print(text)

How to use the C socket API in C++ on z/OS I'm having issues getting the C sockets API to work properly in C++ on z/OS. Although I am including sys/socket.h, I still get compile time errors telling me that AF_INET is not defined. Am I missing something obvious, or is this related to the fact that being on z/OS makes my problems much more complicated? I discovered that there is a #ifdef that I'm hitting. Apparently, z/OS isn't happy unless I define which 'type' of sockets I'm using with: #define _OE_SOCKETS  Now, I personally have no idea what this _OE_SOCKETS is actually for, so if any z/OS sockets programmers are out there (all 3 of you), perhaps you could give me a rundown of how this all works? Test App: #include &lt;sys/socket.h&gt;  int main() {     return AF_INET; }  Compile/Link Output: cxx -Wc,xplink -Wl,xplink -o inet_test inet.C  './inet.C', line 5.16: CCN5274 (S) The name lookup for 'AF_INET' did not find a declaration. CCN0797(I) Compilation failed for file ./inet.C. Object

In [11]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [69]:
# Inicializa o LLM com a chave da API OpenAI
llm = OpenAI(temperature=0.3, openai_api_key=os.getenv("OPENAI_API_KEY"))

In [90]:
# Configura o prompt com ajustes para garantir uma resposta estruturada
qa_prompt = PromptTemplate.from_template(
    "You are a C++ expert and are well familiar with discussions on Stack Overflow. "
    "Answer the following question using the information provided:\n"
    "Question: What are the main benefits and challenges of migrating from C++{previous_version} to C++{current_version}?\n\n"
    "Information provided: {context}\n\n"
    "Please format your answer as follows:\n"
    "### Benefits of Migration\n"
    "1. [Benefit 1]\n"
    "2. [Benefit 2]\n"
    "3. [Benefit 3]\n\n"
    "### Challenges of Migration\n"
    "1. [Challenge 1]\n"
    "2. [Challenge 2]\n"
    "3. [Challenge 3]\n"
)

# Criação do llm_chain
llm_chain = LLMChain(llm=llm, prompt=qa_prompt)

In [14]:
# Carregar o modelo de embedding
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [15]:
# Gerar embeddings em lotes usando processamento paralelo //tentativa de diminuir o tempo de embeddings
def batch_encode_parallel(texts, model, batch_size=256):
    embeddings = []
    try:
        with concurrent.futures.ThreadPoolExecutor() as executor:
            # Gerar embeddings em paralelo
            futures = {executor.submit(model.encode, texts[i:i + batch_size]): i for i in range(0, len(texts), batch_size)}
            for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
                batch_embeddings = future.result()
                embeddings.append(batch_embeddings)
        print("Embeddings gerados com sucesso.")
        return np.vstack(embeddings)
    except Exception as e:
        print(f"Erro ao gerar embeddings: {str(e)}")
        return None

embeddings = batch_encode_parallel(texts, model)

100%|██████████| 151/151 [1:13:53<00:00, 29.36s/it]

Embeddings gerados com sucesso.


In [125]:
# Inicializar o cliente do Qdrant
client = qdrant_client.QdrantClient(":memory:")
collection_name = "relevant_posts"

In [127]:
# Verificar se a coleção já existe e recriar se necessário
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=embeddings.shape[1], distance=Distance.COSINE),
)

<ipython-input-127-acc8e7a40ee1>:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [131]:
# Adicionar documentos ao Qdrant
points = [
    PointStruct(id=i, vector=embedding.tolist(), payload={"text": texts[i]})
    for i, embedding in enumerate(embeddings)
]
client.upsert(collection_name=collection_name, points=points)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [108]:
# Função para realizar a busca e recuperar os documentos mais relevantes
def retrieve(query, client, collection_name, embedding_model, k=10):
    try:
        # Gera o vetor de consulta usando o modelo de embeddings
        query_embedding = embedding_model.encode(query).tolist()  # Utilize o modelo de embeddings aqui

        # Realiza a busca na coleção
        search_result = client.search(
            collection_name=collection_name,
            query_vector=query_embedding,
            limit=k,
            search_params={"hnsw_ef": 100}  # Parâmetro HNSW
        )

        documents = []

        # Adicionar documentos
        for hit in search_result:
            documents.append(Document(page_content=hit.payload["text"]))

        return documents
    except Exception as e:
        print(f"Erro ao buscar documentos: {str(e)}")
        return []  # Retorna uma lista vazia

In [109]:
# Função para filtrar o contexto com base na relevância
def filter_context(context, query, threshold=0.2):
    # Contar a frequência das palavras na consulta
    query_words = query.lower().split()
    context_sentences = context.split('. ')

    # Calcular a relevância de cada sentença
    relevance_scores = []
    for sentence in context_sentences:
        score = sum(Counter(query_words)[word] for word in sentence.lower().split())
        relevance_scores.append((sentence, score))

    # Filtrar sentenças com base na pontuação de relevância
    filtered_context = [sentence for sentence, score in relevance_scores if score >= threshold]

    return ' '.join(filtered_context)

In [110]:
# Função para dividir o contexto em partes menores com validação
def split_context(context, max_len=1000):
    try:
        sentences = re.split(r'(?<=[.!?]) +', context)
        chunks = []
        chunk = ''

        for sentence in sentences:
            if len(chunk) + len(sentence) + 1 <= max_len:
                chunk += sentence + ' '
            else:
                chunks.append(chunk.strip())
                chunk = sentence + ' '

        if chunk:
            chunks.append(chunk.strip())

#        print(f"{len(chunks)} chunks de contexto criados.")
#        return chunks
#    except Exception as e:
#        print(f"Erro ao dividir o contexto: {str(e)}")
#        return []

In [111]:
# Função para remover redundâncias nas respostas
def remove_redundancies(responses):
    unique_responses = list(set(responses))
    return unique_responses

In [115]:
# Função para processar em paralelo os chunks de contexto e gerar respostas
def process_chunk(chunk, llm_chain, previous_version, current_version):
    try:
        # Atualiza as variáveis no prompt
        return llm_chain.run(context=chunk, previous_version=previous_version, current_version=current_version).strip()
    except Exception as e:
        print(f"Erro ao processar chunk: {str(e)}")
        return None

In [116]:
# Função para gerar a resposta com base em uma consulta e documentos recuperados
def generate_answer(query, client, collection_name, llm_chain: LLMChain, embedding_model, previous_version=None, current_version=None):
    try:
        # Realizar busca para encontrar os documentos mais relevantes
        documents = retrieve(query, client, collection_name, embedding_model)

        if not documents:
            return "Nenhum documento relevante foi encontrado para essa consulta."

        # Concatenar o conteúdo dos documentos para usar como contexto
        context = " ".join([doc.page_content for doc in documents])

        # Filtrar o contexto com base na relevância da consulta
        filtered_context = filter_context(context, query)

        # Criar um dicionário para passar as variáveis necessárias
        prompt_variables = {
            "context": filtered_context,
            "previous_version": previous_version,
            "current_version": current_version,
            "query": query
        }

        # Gerar a resposta usando o LLM e o contexto dos documentos filtrados
        responses = []
        chunks = split_context(filtered_context)

        # Passando `previous_version` e `current_version` para `process_chunk`
        for chunk in chunks:
            response = process_chunk(chunk, llm_chain, previous_version, current_version)
            if response:
                responses.append(response)

        # Remover redundâncias nas respostas
        unique_responses = remove_redundancies(responses)

        # Estruturar a resposta final
        final_response = "\n".join(unique_responses)

        return final_response
    except Exception as e:
        print(f"Erro ao gerar resposta: {str(e)}")
        return "Ocorreu um erro ao gerar a resposta."

In [124]:
# Variáveis de versão
previous_version = '20'
current_version = '23'

# Teste de consulta
query = "What are the main benefits and challenges of migrating from c++20 to c++23?"
answer = generate_answer(query, client, collection_name, llm_chain, model, previous_version, current_version)

print(answer)

7 chunks de contexto criados.
### Benefits of Migration
1. Improved language features: C++23 is expected to introduce new language features that can improve code readability, maintainability, and performance.
2. Better compatibility: Migrating to C++23 can ensure compatibility with newer compilers and libraries, making it easier to integrate with other codebases.
3. Enhanced security: C++23 is expected to include security enhancements, making it a more secure language for developing applications.

### Challenges of Migration
1. Learning curve: Migrating to a new version of a language can require learning new syntax and features, which can be challenging for developers who are used to working with older versions.
2. Codebase compatibility: Migrating to C++23 may require making changes to existing codebases, which can be time-consuming and may introduce bugs.
3. Limited support: Since C++23 is still in development, there may be limited resources and support available for troubleshooting 